<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_2_ROC_AUC_Threshold_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Predicting Credit Default: Part 3
## ROC Curves and Threshold Tuning

**Author:** Brad Sheese

---

## Introduction: Comparing Curves, Not Points

In Part 2, we learned that changing the **Decision Threshold** (the point where we say "yes" vs. "no") drastically changes our Precision and Recall. This creates a problem: if I have two models, how do I know which one is better overall if their performance depends on an arbitrary threshold choice?

In this notebook, we introduce the **ROC (Receiver Operating Characteristic) Curve** and the **AUC (Area Under the Curve)** score. These tools allow us to evaluate a model's quality across *every* possible threshold simultaneously. We will also learn how to use **Youden's J Statistic** to find the mathematically "optimal" threshold for a given problem.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Interpret **ROC Curves** and understand the trade-off between **TPR** and **FPR**.
2. Use **AUC** to compare the performance of different models.
3. Understand why **Precision-Recall Curves** are sometimes better than ROC curves for imbalanced data.
4. Calculate **Youden's J** to find the "Goldilocks" decision boundary.

## Problem 1: Setup and Data Loading

As before, we'll train our balanced Logistic Regression model on the South German Credit data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve, f1_score
import warnings
warnings.filterwarnings('ignore')

data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:, 1]
print("Model ready for evaluation.")

## Problem 2: The ROC Curve (TPR vs. FPR)

The **ROC Curve** plots two specific metrics against each other as we slide our threshold from 0 to 1:

1.  **True Positive Rate (TPR)**: Same as **Recall**. "Of all defaults, how many did we catch?"
2.  **False Positive Rate (FPR)**: "Of all good customers, how many did we wrongly flag?" (Note: FPR = 1 - Specificity).

### What to Look For:
- A **Perfect Model** would hug the top-left corner (TPR=1, FPR=0).
- A **Random Model** would be a diagonal line (the chance line).
- The more "arched" the curve is toward the top-left, the better the model is at separating the two classes.

In [ ]:
# Calculate the ROC curve points
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Chance')

plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve: Credit Default Model')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

### Discussion: The AUC Score
The **AUC (Area Under the Curve)** is the single best metric for model comparison. It represents the probability that the model will rank a randomly chosen positive instance higher than a randomly chosen negative one. 

- **0.5**: Useless (Random)
- **0.7 - 0.8**: Acceptable
- **0.8 - 0.9**: Excellent
- **0.9+**: Outstanding (But check for data leakage!)

## Problem 3: ROC vs. Precision-Recall Curves

While ROC curves are standard, they can be over-optimistic on highly imbalanced data. This is because the **False Positive Rate (FPR)** uses the "True Negatives" in its denominator. If you have millions of True Negatives, your FPR will stay low even if you have thousands of False Positives.

The **Precision-Recall (PR) Curve** is often more "honest" for imbalanced data because it ignores True Negatives entirely. Let's compare.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision) # Note: order is (x, y)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR Curve (AUC = {pr_auc:.3f})')
plt.axhline(y=y_test.mean(), color='red', linestyle='--', label='Baseline (% Positives)')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve: Credit Default Model')
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

### Look Closely:
The baseline for an ROC curve is always 0.5. The baseline for a PR curve is the percentage of positive cases in your data (in our case, ~0.3). If your PR curve isn't well above that red line, your model isn't doing much better than guessing!

## Problem 4: Finding the "Goldilocks" Threshold (Youden's J)

Now that we've evaluated the *curve*, how do we pick a single *point* (threshold) to use in production?

One common method is **Youden's J Statistic**. It finds the threshold that maximizes the distance between the ROC curve and the random chance line.

$$J = TPR - FPR$$

Essentially, it finds the threshold that gives you the most "bang for your buck"—maximizing detections while minimizing false alarms.

In [ ]:
# Calculate J for all thresholds on the ROC curve
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]

print(f"--- Youden's J Results ---")
print(f"Best Threshold: {best_threshold:.3f}")
print(f"Max J-Score:    {j_scores[best_idx]:.3f}")
print(f"TPR (Recall) at this point: {tpr[best_idx]:.3f}")
print(f"FPR at this point: {fpr[best_idx]:.3f}")

## Problem 5: Business Cost Sensitivity

While Youden's J is mathematically optimal, it assumes that a False Positive and a False Negative have equal cost. In the real world, they rarely do.

**Assume:**
- A **False Positive** (wrongly denying a loan) costs the bank **$500** in lost interest.
- A **False Negative** (wrongly approving a bad loan) costs the bank **$5,000** in lost principal.

In this case, we would want to move our threshold to be much more sensitive to catch those defaults, even if it means more false alarms.

In [ ]:
test_thresholds = np.arange(0.1, 0.9, 0.05)
costs = []

for t in test_thresholds:
    y_t = (y_proba >= t).astype(int)
    fp = ((y_t == 1) & (y_test == 0)).sum()
    fn = ((y_t == 0) & (y_test == 1)).sum()
    total_cost = (fp * 500) + (fn * 5000)
    costs.append(total_cost)

plt.figure(figsize=(10, 5))
plt.plot(test_thresholds, costs, marker='o', color='crimson')
plt.axvline(x=test_thresholds[np.argmin(costs)], color='green', linestyle='--', label='Cost-Optimal Threshold')
plt.xlabel('Threshold')
plt.ylabel('Total Estimated Cost ($)')
plt.title('Business-Optimal Threshold Selection')
plt.legend()
plt.show()

print(f"Mathematical Optimal (Youden's J): {best_threshold:.3f}")
print(f"Business Optimal (Min Cost):      {test_thresholds[np.argmin(costs)]:.3f}")

## Conclusion
We have now mastered the art of classification evaluation! We know how to:
1.  Use **AUC** to compare models independent of their thresholds.
2.  Use **PR Curves** to check for performance on imbalanced data.
3.  Use **Youden's J** or **Cost Analysis** to pick the right decision boundary for the real world.

In Part 4, we will look at how to compare multiple model types (Trees, SVMs, etc.) and how to extend these concepts to **Multiclass** problems.